# Étape 3 — Modèle Avancé : Decision Tree

Dans ce notebook, on entraîne un **arbre de décision** sur les deux cibles du jeu de Morpion :
- `x_wins` : X gagne-t-il en jeu parfait depuis cet état ?
- `is_draw` : la partie est-elle nulle en jeu parfait depuis cet état ?

L'arbre de décision est un modèle naturellement interprétable : il pose une série de questions sur les cases du plateau et arrive à une décision. C'est un peu comme la façon dont un joueur humain raisonne.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
print("Librairies chargées avec succès.")

## 1. Chargement des données

In [ ]:
# Chargement du fichier CSV généré à l'Étape 0
df = pd.read_csv("../../ressources/dataset_morpion.csv")

# Séparation des features (les 18 cases) et des deux cibles
X = df.drop(columns=['x_wins', 'is_draw'])
y_wins = df['x_wins']
y_draw = df['is_draw']

# Découpage 80% entraînement / 20% test
X_train, X_test, y_train_wins, y_test_wins = train_test_split(X, y_wins, test_size=0.2, random_state=42)
_, _, y_train_draw, y_test_draw = train_test_split(X, y_draw, test_size=0.2, random_state=42)

print(f"Nombre d'échantillons d'entraînement : {X_train.shape[0]}")
print(f"Nombre d'échantillons de test         : {X_test.shape[0]}")

## 2. Baseline — Régression Logistique

On ré-entraîne rapidement la baseline ici pour pouvoir comparer les résultats directement dans ce notebook, sans avoir à consulter un autre fichier.

In [ ]:
# Entraînement de la baseline sur les deux cibles
lr_wins = LogisticRegression(max_iter=1000, random_state=42)
lr_wins.fit(X_train, y_train_wins)

lr_draw = LogisticRegression(max_iter=1000, random_state=42)
lr_draw.fit(X_train, y_train_draw)

# Métriques de la baseline
lr_pred_wins = lr_wins.predict(X_test)
lr_pred_draw = lr_draw.predict(X_test)

print("Baseline — Régression Logistique")
print(f"  Accuracy x_wins  : {accuracy_score(y_test_wins, lr_pred_wins):.2%}")
print(f"  Accuracy is_draw : {accuracy_score(y_test_draw, lr_pred_draw):.2%}")

## 3. Entraînement du Decision Tree

On entraîne deux arbres indépendants, un pour chaque cible.

**Principaux hyperparamètres :**
- `max_depth` : profondeur maximale de l'arbre. Sans limite, l'arbre mémorise les données (surapprentissage). On fixe ici une limite raisonnable.
- `criterion='gini'` : mesure utilisée pour choisir la meilleure question à chaque nœud (impureté de Gini).

In [ ]:
# Modèle pour la victoire de X
dt_wins = DecisionTreeClassifier(
    criterion='gini',  # Critère de séparation des nœuds
    max_depth=8,       # Profondeur maximale pour limiter le surapprentissage
    random_state=42
)

print("Entraînement de l'arbre de décision (Wins)...")
dt_wins.fit(X_train, y_train_wins)

# Modèle pour le match nul — mêmes paramètres pour cohérence
dt_draw = DecisionTreeClassifier(
    criterion='gini',
    max_depth=8,
    random_state=42
)

print("Entraînement de l'arbre de décision (Draws)...")
dt_draw.fit(X_train, y_train_draw)

print(f"\nProfondeur réelle de l'arbre x_wins  : {dt_wins.get_depth()}")
print(f"Profondeur réelle de l'arbre is_draw : {dt_draw.get_depth()}")

## 4. Évaluation — x_wins

In [ ]:
# Prédictions sur le jeu de test
y_pred_wins = dt_wins.predict(X_test)

print("\n--- RÉSULTATS DECISION TREE (VICTOIRES X) ---")
print(f"Précision globale (Accuracy) : {accuracy_score(y_test_wins, y_pred_wins):.2%}")
print("\nRapport de classification :")
print(classification_report(y_test_wins, y_pred_wins))

# Matrice de confusion
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_wins, y_pred_wins)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Matrice de Confusion : Decision Tree — x_wins')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

## 5. Évaluation — is_draw

In [ ]:
# Prédictions sur le jeu de test
y_pred_draw = dt_draw.predict(X_test)

print("\n--- RÉSULTATS DECISION TREE (MATCHS NULS) ---")
print(f"Précision globale (Accuracy) : {accuracy_score(y_test_draw, y_pred_draw):.2%}")
print("\nRapport de classification :")
print(classification_report(y_test_draw, y_pred_draw))

# Matrice de confusion
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_draw, y_pred_draw)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False)
plt.title('Matrice de Confusion : Decision Tree — is_draw')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

## 6. Comparaison avec la Baseline

On compare maintenant le Decision Tree à la Régression Logistique sur les deux cibles.

In [ ]:
# Tableau récapitulatif des scores
results = {
    'Modèle': ['Régression Logistique (baseline)', 'Decision Tree'],
    'Accuracy x_wins': [
        accuracy_score(y_test_wins, lr_pred_wins),
        accuracy_score(y_test_wins, y_pred_wins)
    ],
    'F1 x_wins': [
        f1_score(y_test_wins, lr_pred_wins, average='weighted'),
        f1_score(y_test_wins, y_pred_wins, average='weighted')
    ],
    'Accuracy is_draw': [
        accuracy_score(y_test_draw, lr_pred_draw),
        accuracy_score(y_test_draw, y_pred_draw)
    ],
    'F1 is_draw': [
        f1_score(y_test_draw, lr_pred_draw, average='weighted'),
        f1_score(y_test_draw, y_pred_draw, average='weighted')
    ],
}

df_results = pd.DataFrame(results).set_index('Modèle')
print(df_results.to_string(float_format='{:.4f}'.format))

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cible, cols in zip(
    axes,
    ['x_wins', 'is_draw'],
    [['Accuracy x_wins', 'F1 x_wins'], ['Accuracy is_draw', 'F1 is_draw']]
):
    df_plot = df_results[cols].copy()
    df_plot.columns = ['Accuracy', 'F1']
    df_plot.T.plot(kind='bar', ax=ax, alpha=0.85, rot=0)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'Comparaison — {cible}', fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.suptitle('Decision Tree vs Baseline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Visualisation de l'arbre (3 premiers niveaux)

L'arbre complet serait illisible (jusqu'à 8 niveaux). On affiche ici les **3 premiers niveaux** pour comprendre les décisions principales prises par le modèle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 7))

for ax, model, title in zip(
    axes,
    [dt_wins, dt_draw],
    ['Arbre de décision — x_wins', 'Arbre de décision — is_draw']
):
    plot_tree(
        model,
        max_depth=3,
        feature_names=X.columns.tolist(),
        class_names=['Non', 'Oui'],
        filled=True,
        rounded=True,
        fontsize=8,
        ax=ax
    )
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Importance des features

Pour chaque case du plateau, on visualise l'importance que l'arbre lui a accordée — séparément pour les occupations X (`ci_x`) et O (`ci_o`).

Une importance élevée signifie que cette case est souvent utilisée pour prendre une décision dans l'arbre.

In [ ]:
feature_names = X.columns.tolist()

for model, title in [
    (dt_wins, 'Importance des features — Decision Tree : x_wins'),
    (dt_draw, 'Importance des features — Decision Tree : is_draw'),
]:
    imp = dict(zip(feature_names, model.feature_importances_))
    board_x = np.array([imp.get(f'c{i}_x', 0) for i in range(9)]).reshape(3, 3)
    board_o = np.array([imp.get(f'c{i}_o', 0) for i in range(9)]).reshape(3, 3)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, board, label, cmap in zip(
        axes,
        [board_x, board_o],
        ['Cases occupées par X (ci_x)', 'Cases occupées par O (ci_o)'],
        ['Greens', 'Blues']
    ):
        sns.heatmap(
            board, annot=True, fmt='.4f', cmap=cmap, ax=ax,
            linewidths=0.5, linecolor='gray',
            xticklabels=['Col 0', 'Col 1', 'Col 2'],
            yticklabels=['Ligne 0', 'Ligne 1', 'Ligne 2'],
            vmin=0
        )
        ax.set_title(label)

    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Export des modèles

On sauvegarde les deux modèles entraînés au format `.pkl` pour les réutiliser dans l'interface jouable à l'Étape 4.

In [ ]:
joblib.dump(dt_wins, '../../ressources/model_dt_wins.pkl')
joblib.dump(dt_draw, '../../ressources/model_dt_draw.pkl')

print("Fichiers .pkl générés avec succès !")
print("  → ressources/model_dt_wins.pkl")
print("  → ressources/model_dt_draw.pkl")